# 2.2 Runtime Context — Per-Request User Data

## What you will learn in this notebook

- What **runtime context** is and why it's different from state and memory
- How to define a **context schema** using a Python dataclass
- How to pass context values at **call time** (per-request, not globally)
- How tools access context via **`ToolRuntime`** without it being a tool argument
- Why context is the right pattern for **per-user, per-request data**

---

### Three ways to give an agent data — when to use which

| Mechanism | What it is | Persists? | User sees it? | Best for |
|---|---|---|---|---|
| **Messages** | Conversation history | Yes (via checkpointer) | Yes | Back-and-forth dialogue |
| **State** | Custom fields on the graph state | Yes (via checkpointer) | No | Derived data the agent computes |
| **Runtime Context** | Data injected per `.invoke()` call | No | No | Per-request user profile data |

### What is runtime context?

**Runtime context** is data you pass to the agent at invocation time that is available to tools but is NOT part of the conversation history and does NOT get stored in state.

Think of it like a **request header** in a web API:
- A web request carries `Authorization: Bearer <token>` in the header
- The handler reads the token to identify the user
- The token is not part of the request body; the user never typed it

Similarly, runtime context is like a **session object** your server injects at request time:
```python
# In a FastAPI endpoint:
agent.invoke(user_message, context=UserContext(user_id=current_user.id, plan="pro"))
```

The tools read `runtime.context.user_id` — the user never said their ID, and it's never stored in the conversation.

### Why not just put it in the system prompt?

You could put `favourite_colour: blue` in the system prompt — but:
- System prompts are part of the conversation, consuming tokens every call
- System prompts are static strings; context objects are typed and validated
- Sensitive data (user IDs, payment info) shouldn't be in prompts

In [ ]:
# ============================================================
# CELL 1: Load Environment Variables
# ============================================================

from dotenv import load_dotenv

load_dotenv()

In [ ]:
# ============================================================
# CELL 2: Define the Context Schema
# ============================================================
# The context schema is a Python dataclass that defines WHAT data
# can be passed to the agent at invocation time.
#
# We use @dataclass for simplicity — LangChain also accepts Pydantic
# models if you need field validation.
#
# Default values are important:
#   favourite_colour: str = "blue"  → if not supplied, defaults to blue
#   least_favourite_colour: str = "yellow"
#
# In a real app this might be:
#   @dataclass
#   class UserContext:
#       user_id: str
#       subscription_plan: str = "free"
#       preferred_language: str = "en"
#       timezone: str = "UTC"

from dataclasses import dataclass

@dataclass
class ColourContext:
    favourite_colour: str = "blue"      # Default if not overridden
    least_favourite_colour: str = "yellow"

In [ ]:
# ============================================================
# CELL 3: Create an Agent That Accepts Context (No Tools Yet)
# ============================================================
# context_schema=ColourContext tells LangChain:
#   "When this agent is invoked, expect an optional context=
#    argument that is an instance of ColourContext"
#
# Without context_schema, passing context= to .invoke() is ignored.
# With it, the context object becomes available to tools via ToolRuntime.
#
# At this stage (no tools), the context is declared but the agent
# can't read it — it doesn't have a tool to access it.
# This is just the declaration step.

from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    context_schema=ColourContext   # Declare what context type to expect
)

In [ ]:
# ============================================================
# CELL 4: Invoke With Context — But No Tool to Read It
# ============================================================
# We pass context=ColourContext() — the default values (blue/yellow).
#
# The model CANNOT read this context directly — it doesn't appear
# in the conversation history or system prompt. The agent has no
# tool to access it, so it will say it doesn't know the colour.
#
# This demonstrates the important point: context is data for TOOLS,
# not for the model. The model needs a tool to surface the context.
# See Section 2 for that.

from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()   # Inject context at call time
)

In [ ]:
# ============================================================
# CELL 5: Inspect — The Agent Doesn't Know (No Tool)
# ============================================================
# The response will say something like "I don't have information
# about your preferences." — because the context is injected but
# inaccessible without a tool.

from pprint import pprint

pprint(response)

---
## Section 2 — Accessing Context From Tools

Tools access context via `ToolRuntime` — a special parameter that LangChain injects automatically when the tool is called.

### The `ToolRuntime` pattern

```python
@tool
def my_tool(user_arg: str, runtime: ToolRuntime) -> str:
    # runtime is injected by LangChain — the model never passes it
    # runtime.context is the ColourContext object we passed to .invoke()
    data = runtime.context.some_field
    return data
```

**Critical**: `runtime: ToolRuntime` must be the **last parameter** and is NOT included in the tool's schema. The model never knows about it — it can't pass it even if it wanted to. LangChain injects it silently.

This is how you pass server-side secrets (user IDs, API tokens) to tools without the model ever seeing them.

In [ ]:
# ============================================================
# CELL 6: Define Tools That Read From Context
# ============================================================
# ToolRuntime is the gateway to context inside a tool.
#
# Key rules for ToolRuntime:
#   1. It must be typed as `runtime: ToolRuntime` exactly
#   2. It must be the LAST parameter in the function signature
#   3. It does NOT appear in the tool's argument schema
#      → The model cannot and does not pass this argument
#      → LangChain injects it automatically at tool call time
#
# runtime.context  → the ColourContext object from .invoke(context=...)
# runtime.context.favourite_colour  → "blue" (or whatever was passed)
#
# Notice: these tools take NO user-facing arguments (no `x: float`, etc.)
# They purely read from context. The model calls them with empty args
# and gets back the context value.

from langchain.tools import tool, ToolRuntime

@tool
def get_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the favourite colour of the user"""
    # runtime is injected by LangChain — not passed by the model
    return runtime.context.favourite_colour

@tool
def get_least_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the least favourite colour of the user"""
    return runtime.context.least_favourite_colour

In [ ]:
# ============================================================
# CELL 7: Create the Agent With Context-Reading Tools
# ============================================================

agent = create_agent(
    model="gpt-5-nano",
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColourContext   # Must declare the schema to use context
)

In [ ]:
# ============================================================
# CELL 8: Invoke With Default Context (blue)
# ============================================================
# ColourContext() uses default values: favourite_colour="blue"
#
# The agent will:
#   1. See the question about favourite colour
#   2. Decide to call get_favourite_colour tool
#   3. LangChain injects runtime.context = ColourContext()
#   4. Tool returns "blue"
#   5. Agent replies: "Your favourite colour is blue"

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()   # Default: favourite_colour="blue"
)

pprint(response)

In [ ]:
# ============================================================
# CELL 9: Invoke With Custom Context (green) — Same Agent!
# ============================================================
# This is the power of runtime context: the SAME agent and the SAME
# tools, but different context per call = different output per user.
#
# ColourContext(favourite_colour="green") overrides the default.
# The tool reads runtime.context.favourite_colour → "green"
# and the agent replies accordingly.
#
# In production: each HTTP request would pass a different UserContext
# (different user_id, subscription plan, locale, etc.) and the same
# agent would serve all users correctly without any shared state
# or per-user agent instances.

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(favourite_colour="green")  # Override for this request
)

pprint(response)

---
## Summary

| Concept | Key takeaway |
|---|---|
| **Runtime context** | Per-request data injected at `.invoke()` time — not in history, not in state |
| **`context_schema=`** | Declares the expected context type on `create_agent()` |
| **`context=`** | Passes an instance at `.invoke()` time |
| **`ToolRuntime`** | Injected automatically into tools as the last parameter — model never sees it |
| **`runtime.context`** | Access the context object inside a tool |
| **Isolation** | Same agent, different context per call = correct per-user behaviour |

### When to use runtime context vs alternatives

```
User-typed data ("my name is Seán")  →  Messages / Memory
Agent-computed data ("user is premium")  →  State
Server-side user profile (user_id, plan)  →  Runtime Context  ← this notebook
```

### Next steps
- **2.2 State** — writing and reading custom fields on the agent's graph state
- **2.3 Multi-Agent** — orchestrating multiple specialised agents from a main agent